In [14]:
from pathlib import Path
from importlib.metadata import distributions, version
import hashlib
import platform
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from mlflow import MlflowClient
from mlflow.models import infer_signature

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    make_scorer,
    silhouette_score,
    ConfusionMatrixDisplay,
)

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import FEATURES
from src.training import create_candidates

DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"
METRICS_DIRECTORY = PROJECT_ROOT / "reports" / "metrics"

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("diabetes-profile-ai")

train_path = DATA_DIRECTORY / "train_labeled.csv"
test_path = DATA_DIRECTORY / "test_labeled.csv"

train_data = pd.read_csv(train_path, index_col="source_row")
test_data = pd.read_csv(test_path, index_col="source_row")

X_train = train_data.loc[:, FEATURES].astype(float)
y_train = train_data["Cluster"].astype(int)

X_test = test_data.loc[:, FEATURES].astype(float)
y_test = test_data["Cluster"].astype(int)

if not train_data.index.intersection(test_data.index).empty:
    raise ValueError("Training and test rows overlap.")

clustering_bundle = joblib.load(
    MODELS_DIRECTORY / "clustering_bundle.joblib"
)
final_bundle = joblib.load(
    MODELS_DIRECTORY / "classification_final.joblib"
)

library_versions = {
    package.metadata["Name"]: package.version
    for package in distributions()
    if package.metadata["Name"]
}

dataset_metadata = {
    "train_sha256": hashlib.sha256(train_path.read_bytes()).hexdigest(),
    "test_sha256": hashlib.sha256(test_path.read_bytes()).hexdigest(),
    "train_rows": len(train_data),
    "test_rows": len(test_data),
    "features": list(FEATURES),
    "target": "Cluster",
    "input_dtypes": X_train.dtypes.astype(str).to_dict(),
}


def log_reproducibility():
    mlflow.log_dict(dataset_metadata, "metadata/dataset.json")
    mlflow.log_dict(library_versions, "metadata/library_versions.json")
    mlflow.log_dict({"features": list(FEATURES)}, "metadata/features.json")
    mlflow.set_tags({
        "python_version": platform.python_version(),
        "target_type": "kmeans_cluster",
        "clinical_validation": "not_established",
    })

    split_metadata = DATA_DIRECTORY / "split_metadata.json"
    if split_metadata.exists():
        mlflow.log_artifact(str(split_metadata), artifact_path="metadata")


print("MLflow version:", mlflow.__version__)
print("Training:", X_train.shape)
print("Test:", X_test.shape)
print("Tracking server:", mlflow.get_tracking_uri())

MLflow version: 3.17.0
Training: (614, 7)
Test: (154, 7)
Tracking server: http://127.0.0.1:5000


In [15]:
clustering_pipeline = clustering_bundle["pipeline"]

preprocessor = clustering_pipeline.steps[0][1]
kmeans = clustering_pipeline.steps[-1][1]

# Apply every step before K-Means.
X_train_scaled = clustering_pipeline[:-1].transform(X_train)
cluster_predictions = clustering_pipeline.predict(X_train)

np.testing.assert_array_equal(
    cluster_predictions,
    y_train.to_numpy(),
)
np.testing.assert_array_equal(
    clustering_pipeline.predict(X_test),
    y_test.to_numpy(),
)

with mlflow.start_run(run_name="KMeans_saved_model"):
    log_reproducibility()

    mlflow.log_params({
        "k": int(kmeans.n_clusters),
        "random_state": kmeans.random_state,
        "n_init": kmeans.n_init,
    })

    mlflow.log_metrics({
        "train_inertia": float(kmeans.inertia_),
        "train_silhouette": float(
            silhouette_score(X_train_scaled, cluster_predictions)
        ),
    })

    scaler = preprocessor.named_steps["scaler"]
    mlflow.log_params({
        f"scaler__{name}": value
        for name, value in scaler.get_params().items()
    })

    mlflow.log_dict(
        {
            str(key): value
            for key, value in clustering_bundle["risk_mapping"].items()
        },
        "metadata/risk_mapping.json",
    )

    mlflow.log_artifact(
        str(MODELS_DIRECTORY / "clustering_bundle.joblib"),
        artifact_path="models",
    )

    # Save the fitted components separately for the project requirements.
    tracking_directory = MODELS_DIRECTORY / "tracking"
    tracking_directory.mkdir(exist_ok=True)

    for name, component in [
        ("kmeans", kmeans),
        ("scaler", scaler),
    ]:
        component_path = tracking_directory / f"{name}.joblib"
        joblib.dump(component, component_path)
        mlflow.log_artifact(str(component_path), artifact_path="models")

print("Clustering run recorded.")

🏃 View run KMeans_saved_model at: http://127.0.0.1:5000/#/experiments/1/runs/fe2f981da3944e9b892a02c87817323f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Clustering run recorded.


In [16]:
class_counts = y_train.value_counts()

if len(class_counts) < 2 or class_counts.min() < 2:
    raise ValueError("Not enough examples for stratified cross-validation.")

cv = StratifiedKFold(
    n_splits=min(5, int(class_counts.min())),
    shuffle=True,
    random_state=42,
)
folds = list(cv.split(X_train, y_train))

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(
        precision_score, average="macro", zero_division=0
    ),
    "recall": make_scorer(
        recall_score, average="macro", zero_division=0
    ),
    "f1": make_scorer(
        f1_score, average="macro", zero_division=0
    ),
}

for name, pipeline in create_candidates().items():
    print("Tracking:", name)

    with mlflow.start_run(run_name=name):
        log_reproducibility()
        mlflow.set_tag("evaluation", "training_cross_validation")
        mlflow.log_param("cv_folds", len(folds))

        mlflow.log_params({
            f"classifier__{key}": value
            for key, value in
            pipeline.named_steps["classifier"].get_params().items()
        })

        sampler = pipeline.named_steps["sampler"]
        mlflow.log_param("sampler", str(sampler))

        if hasattr(sampler, "get_params"):
            mlflow.log_params({
                f"sampler__{key}": value
                for key, value in sampler.get_params().items()
            })

        scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=folds,
            scoring=scoring,
            return_train_score=True,
            return_estimator=True,
            n_jobs=1,
            error_score="raise",
        )

        mlflow.log_metrics({
            "cv_accuracy": float(scores["test_accuracy"].mean()),
            "cv_precision_macro": float(scores["test_precision"].mean()),
            "cv_recall_macro": float(scores["test_recall"].mean()),
            "cv_f1_macro": float(scores["test_f1"].mean()),
            "cv_f1_std": float(scores["test_f1"].std()),
            "train_f1_macro": float(scores["train_f1"].mean()),
        })

        predictions = np.empty(len(y_train), dtype=int)

        for estimator, (_, validation_indices) in zip(
            scores["estimator"], folds
        ):
            predictions[validation_indices] = estimator.predict(
                X_train.iloc[validation_indices]
            )

        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_train,
            predictions,
            cmap="Blues",
            colorbar=False,
            ax=ax,
        )
        ax.set_title(f"{name}\nOut-of-fold predictions")
        fig.tight_layout()

        mlflow.log_figure(fig, "figures/cv_confusion_matrix.png")
        plt.close(fig)

print("All six baseline runs recorded.")

Tracking: Logistic Regression | no sampling
🏃 View run Logistic Regression | no sampling at: http://127.0.0.1:5000/#/experiments/1/runs/b918c343c65a4d8182eed9f3580da3aa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Tracking: Logistic Regression | oversampling
🏃 View run Logistic Regression | oversampling at: http://127.0.0.1:5000/#/experiments/1/runs/8fb5011848e54fd2b6682349ad7cdb46
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Tracking: Random Forest | no sampling
🏃 View run Random Forest | no sampling at: http://127.0.0.1:5000/#/experiments/1/runs/3e5be0384d084ac8bfe6f5788c1ca1ce
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Tracking: Random Forest | oversampling
🏃 View run Random Forest | oversampling at: http://127.0.0.1:5000/#/experiments/1/runs/be20317d338c4d6e9896343148f20676
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Tracking: SVC | no sampling
🏃 View run SVC | no sampling at: http://127.0.0.1:5000/#/experiments/1/runs/

In [18]:
from src.optimization import create_logistic_search
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import RandomOverSampler

tuning_path = METRICS_DIRECTORY / "logistic_tuning.csv"

if not tuning_path.exists():
    class_counts = y_train.value_counts()

    if len(class_counts) < 2 or class_counts.min() < 2:
        raise ValueError("Not enough examples for cross-validation.")

    cv = StratifiedKFold(
        n_splits=min(5, int(class_counts.min())),
        shuffle=True,
        random_state=42,
    )

    tuning_folds = list(cv.split(X_train, y_train))

    tracking_search = create_logistic_search(tuning_folds)
    tracking_search.fit(X_train, y_train)

    results = pd.DataFrame(tracking_search.cv_results_)

    tuning_results = pd.DataFrame({
        "C": results["param_classifier__C"].astype(float),
        "Sampling": [
            "oversampling"
            if isinstance(sampler, RandomOverSampler)
            else "no sampling"
            for sampler in results["param_sampler"]
        ],
        "Train_F1": results["mean_train_score"],
        "CV_F1": results["mean_test_score"],
        "CV_F1_std": results["std_test_score"],
    }).sort_values("CV_F1", ascending=False).reset_index(drop=True)

    tuning_path.parent.mkdir(parents=True, exist_ok=True)
    tuning_results.to_csv(tuning_path, index=False)

    print("Created:", tuning_path)
else:
    print("File already exists:", tuning_path)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Created: /home/bsar/Desktop/diabetes-profile-ai/reports/metrics/logistic_tuning.csv


In [20]:
final_pipeline = final_bundle["pipeline"]

# Keep application metadata inside the serialized pipeline.
final_pipeline.feature_list_ = list(FEATURES)
final_pipeline.risk_mapping_ = {
    int(key): value
    for key, value in final_bundle["risk_mapping"].items()
}

tuning_results = pd.read_csv(
    METRICS_DIRECTORY / "logistic_tuning.csv"
)


requirements = [
    f"{package}=={version(package)}"
    for package in [
        "mlflow",
        "scikit-learn",
        "imbalanced-learn",
        "pandas",
        "numpy",
        "scipy",
        "joblib",
        "cloudpickle",
    ]
]

with mlflow.start_run(run_name="Logistic_Regression_final") as run:
    final_run_id = run.info.run_id
    log_reproducibility()

    mlflow.log_params(final_bundle["best_parameters"])
    mlflow.log_param("selected_k", final_bundle["selected_k"])
    mlflow.log_metric("selected_cv_f1_macro", final_bundle["cv_f1"])

    mlflow.log_artifact(
        str(METRICS_DIRECTORY / "logistic_tuning.csv"),
        artifact_path="tuning",
    )

    # Import each previously evaluated search configuration.
    for index, row in tuning_results.iterrows():
        with mlflow.start_run(
            run_name=f"tuning_candidate_{index + 1}",
            nested=True,
        ):
            mlflow.set_tag("source", "saved_grid_search_results")
            mlflow.log_params({
                "C": float(row["C"]),
                "sampling": row["Sampling"],
            })
            mlflow.log_metrics({
                "train_f1_macro": float(row["Train_F1"]),
                "cv_f1_macro": float(row["CV_F1"]),
                "cv_f1_std": float(row["CV_F1_std"]),
            })

    for split_name, X, y in [
        ("train", X_train, y_train),
        ("test", X_test, y_test),
    ]:
        predictions = final_pipeline.predict(X)

        mlflow.log_metrics({
            f"{split_name}_accuracy": accuracy_score(y, predictions),
            f"{split_name}_precision_macro": precision_score(
                y, predictions, average="macro", zero_division=0
            ),
            f"{split_name}_recall_macro": recall_score(
                y, predictions, average="macro", zero_division=0
            ),
            f"{split_name}_f1_macro": f1_score(
                y, predictions, average="macro", zero_division=0
            ),
        })

    test_predictions = final_pipeline.predict(X_test)

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        test_predictions,
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title("Final model — Test set")
    fig.tight_layout()
    mlflow.log_figure(fig, "figures/test_confusion_matrix.png")
    plt.close(fig)

    mlflow.log_dict(
        {
            str(key): value
            for key, value in final_pipeline.risk_mapping_.items()
        },
        "metadata/risk_mapping.json",
    )

    model_info = mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        name="pipeline",
        serialization_format="skops",
        skops_trusted_types=[
            "imblearn.over_sampling._random_over_sampler.RandomOverSampler",
            "imblearn.pipeline.Pipeline",
            "numpy.dtype",
            "src.preprocessing.prepare_features",
        ],
        signature=infer_signature(
            X_train,
            final_pipeline.predict(X_train),
        ),
        code_paths=[str(PROJECT_ROOT / "src")],
        pip_requirements=[
            *requirements,
            f"skops=={version('skops')}",
        ],
    )

print("Final run ID:", final_run_id)
print("Logged model:", model_info.model_uri)

🏃 View run tuning_candidate_1 at: http://127.0.0.1:5000/#/experiments/1/runs/d63dcdd30b1646fc8532d39a07973411
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run tuning_candidate_2 at: http://127.0.0.1:5000/#/experiments/1/runs/6dcacc2d19f04e5583a9f66bb5e29d3a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run tuning_candidate_3 at: http://127.0.0.1:5000/#/experiments/1/runs/7cc815bfff944e10a18228e416914c2d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run tuning_candidate_4 at: http://127.0.0.1:5000/#/experiments/1/runs/10521621c3b749ccaa7c05a6bd49e39b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run tuning_candidate_5 at: http://127.0.0.1:5000/#/experiments/1/runs/7bf85722239a4451afa89dea22769299
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run tuning_candidate_6 at: http://127.0.0.1:5000/#/experiments/1/runs/fba0e7f5c1e44d6691f0c4895025fb09
🧪 View experiment at: http://127.0.0.1:5

In [ ]:
MODEL_NAME = "diabetes-profile-classifier"

registered_version = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=MODEL_NAME,
)

version_uri = (
    f"models:/{MODEL_NAME}/{registered_version.version}"
)

registry_pipeline = mlflow.sklearn.load_model(version_uri)

np.testing.assert_array_equal(
    final_pipeline.predict(X_test),
    registry_pipeline.predict(X_test),
)

assert (
    registry_pipeline.risk_mapping_
    == final_pipeline.risk_mapping_
)

client = MlflowClient()

client.set_model_version_tag(
    name=MODEL_NAME,
    version=registered_version.version,
    key="validation",
    value="cluster_prediction_test_evaluated",
)

client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=registered_version.version,
)

print("Registered model:", MODEL_NAME)
print("Version:", registered_version.version)
print("Registry verification passed.")
print("Application model URI:", f"models:/{MODEL_NAME}@champion")


# mlflow server \
#   --host 127.0.0.1 \
#   --port 5000 \
#   --backend-store-uri sqlite:///mlflow_data/mlflow.db \
#   --artifacts-destination ./mlflow_data/artifacts

Successfully registered model 'diabetes-profile-classifier'.
2026/10/10 13:11:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabetes-profile-classifier, version 1
Created version '1' of model 'diabetes-profile-classifier'.


Registered model: diabetes-profile-classifier
Version: 1
Registry verification passed.
Application model URI: models:/diabetes-profile-classifier@champion
